# Ejercicio 4: Agrupamiento de Clientes según Comportamiento de Compra
**Asignatura:** Análisis de Datos  
**Equipo:** 6  
**Dataset:** Mall Customers Dataset  
**Modelos:** K-Means, DBSCAN, Clustering Jerárquico (Agglomerative)

---
## Descripción del Problema
El objetivo es segmentar clientes de un centro comercial usando **aprendizaje no supervisado** a partir de características demográficas (edad, género) y de consumo (ingresos anuales, puntuación de gasto). Los segmentos obtenidos permiten diseñar estrategias de marketing personalizadas para cada perfil de cliente.

## 1. Instalación e Importación de Librerías

In [ ]:
!pip install scikit-learn pandas numpy matplotlib seaborn scipy -q

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.decomposition import PCA
from scipy.cluster.hierarchy import dendrogram, linkage

print('✅ Librerías importadas correctamente')

## 2. Carga del Dataset

**Dataset requerido:** `Mall_Customers.csv` del [Mall Customers – Kaggle](https://www.kaggle.com/datasets/vjchoudhary7/customer-segmentation-tutorial-in-python)

**Pasos para obtenerlo:**
1. Ir a Kaggle → buscar *Mall Customer Segmentation Data*
2. Descargar el archivo `Mall_Customers.csv`
3. Ejecutar la celda siguiente → clic en **Elegir archivos** → seleccionar `Mall_Customers.csv`

> ℹ️ Este archivo es muy liviano (~5 KB, 200 filas). La subida es instantánea.

In [ ]:
# ============================================================
# OPCIÓN A: Subir Mall_Customers.csv manualmente
# ============================================================
from google.colab import files
import io

print('📂 Selecciona el archivo Mall_Customers.csv desde tu computador:')
uploaded = files.upload()   # Se abre el selector de archivos

filename = list(uploaded.keys())[0]
df = pd.read_csv(io.BytesIO(uploaded[filename]))
print(f'✅ Archivo cargado: {filename}')
print(f'   Shape: {df.shape}')
print(f'   Columnas: {list(df.columns)}')
df.head(10)

In [ ]:
# ============================================================
# Verificación y normalización de nombres de columnas
# (ejecutar siempre después de cargar)
# ============================================================

# El dataset original puede tener variaciones en nombres de columnas.
# Estandarizamos para que el resto del notebook funcione sin cambios.
rename_map = {
    'Gender': 'Genre',
    'Annual Income (k$)': 'Annual Income (k$)',
    'Spending Score (1-100)': 'Spending Score (1-100)'
}
df = df.rename(columns=rename_map)

# Verificar nulos
print('Valores nulos por columna:')
print(df.isnull().sum())
print(f'\nShape final: {df.shape}')
print(df.dtypes)

In [ ]:
# ============================================================
# OPCIÓN B: Cargar desde Google Drive
# ============================================================
# from google.colab import drive
# drive.mount('/content/drive')
# RUTA = '/content/drive/MyDrive/Mall_Customers.csv'  # ← ajustar
# df = pd.read_csv(RUTA)

print('ℹ️  Opción B disponible (comentada). Descomenta si prefieres cargar desde Drive.')

## 3. Análisis Exploratorio de Datos (EDA)

In [ ]:
print('=== Información del Dataset ===')
print(df.info())
print('\n=== Estadísticas Descriptivas ===')
print(df.describe())
print('\n=== Valores nulos ===')
print(df.isnull().sum())

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

numericas = ['Age', 'Annual Income (k$)', 'Spending Score (1-100)']
colores = ['#3498db', '#e67e22', '#9b59b6']

# Histogramas
for i, (col, color) in enumerate(zip(numericas, colores)):
    axes[0, i].hist(df[col], bins=20, color=color, edgecolor='black', alpha=0.8)
    axes[0, i].set_title(f'Distribución: {col}', fontweight='bold')
    axes[0, i].set_xlabel(col)
    axes[0, i].set_ylabel('Frecuencia')
    axes[0, i].axvline(df[col].mean(), color='red', linestyle='--', label=f'Media: {df[col].mean():.1f}')
    axes[0, i].legend()

# Scatter plots clave
scatter_configs = [
    ('Annual Income (k$)', 'Spending Score (1-100)', 'Ingreso vs Gasto'),
    ('Age', 'Spending Score (1-100)', 'Edad vs Gasto'),
    ('Age', 'Annual Income (k$)', 'Edad vs Ingreso'),
]
for i, (x, y, titulo) in enumerate(scatter_configs):
    axes[1, i].scatter(df[x], df[y], alpha=0.6, c=colores[i], edgecolors='white', s=50)
    axes[1, i].set_xlabel(x)
    axes[1, i].set_ylabel(y)
    axes[1, i].set_title(titulo, fontweight='bold')
    axes[1, i].grid(True, alpha=0.3)

plt.suptitle('EDA – Mall Customers Dataset', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_clientes.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Distribución por género
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

df['Genre'].value_counts().plot(kind='pie', ax=axes[0], autopct='%1.1f%%',
                                 colors=['#3498db','#e91e8c'], startangle=90)
axes[0].set_title('Distribución por Género', fontweight='bold')
axes[0].set_ylabel('')

for genero, color in zip(['Male', 'Female'], ['#3498db', '#e91e8c']):
    sub = df[df['Genre'] == genero]
    axes[1].scatter(sub['Annual Income (k$)'], sub['Spending Score (1-100)'],
                   label=genero, alpha=0.6, color=color, s=50)
axes[1].set_xlabel('Annual Income (k$)')
axes[1].set_ylabel('Spending Score (1-100)')
axes[1].set_title('Ingreso vs Gasto por Género', fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Correlación
corr = df[numericas].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', ax=axes[2],
            center=0, square=True, linewidths=0.5)
axes[2].set_title('Mapa de Correlación', fontweight='bold')

plt.tight_layout()
plt.savefig('genero_correlacion.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Preprocesamiento

In [ ]:
# Codificación de género y selección de features
le = LabelEncoder()
df['Genre_encoded'] = le.fit_transform(df['Genre'])

# Features para clustering
features = ['Age', 'Annual Income (k$)', 'Spending Score (1-100)']
X = df[features].copy()

# Estandarización
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled_df = pd.DataFrame(X_scaled, columns=features)

print('Features utilizadas:', features)
print(f'Shape: {X_scaled.shape}')
print('\nEstadísticas post-estandarización:')
print(X_scaled_df.describe().round(3))

## 5. Selección del Número Óptimo de Clusters (Método del Codo)

In [ ]:
inercias = []
silhouettes = []
k_range = range(2, 11)

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    inercias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, labels))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(list(k_range), inercias, 'bo-', markersize=8, linewidth=2)
axes[0].set_xlabel('Número de Clusters (k)', fontsize=12)
axes[0].set_ylabel('Inercia (WCSS)', fontsize=12)
axes[0].set_title('Método del Codo – K-Means', fontsize=14, fontweight='bold')
axes[0].grid(True, alpha=0.4)
axes[0].axvline(5, color='red', linestyle='--', alpha=0.7, label='k óptimo = 5')
axes[0].legend()

axes[1].plot(list(k_range), silhouettes, 'go-', markersize=8, linewidth=2)
axes[1].set_xlabel('Número de Clusters (k)', fontsize=12)
axes[1].set_ylabel('Silhouette Score', fontsize=12)
axes[1].set_title('Silhouette Score vs. k', fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.4)
k_opt = list(k_range)[np.argmax(silhouettes)]
axes[1].axvline(k_opt, color='red', linestyle='--', alpha=0.7, label=f'k óptimo = {k_opt}')
axes[1].legend()

plt.tight_layout()
plt.savefig('codo_silhouette.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'K óptimo por Silhouette: {k_opt} (score={max(silhouettes):.3f})')

## 6. Aplicación de Modelos de Clustering

### Justificación de los Algoritmos
- **K-Means:** El más utilizado para segmentación de clientes. Requiere especificar k, funciona mejor con clusters esféricos y datos estandarizados. Eficiente y fácil de interpretar por equipos de marketing.
- **DBSCAN:** No requiere definir k a priori. Identifica outliers como ruido, ideal cuando existen clientes atípicos. Sin embargo, es sensible a los hiperparámetros `eps` y `min_samples`.
- **Clustering Jerárquico (Agglomerative):** Construye un dendrograma que permite visualizar la jerarquía de grupos. Útil cuando no se sabe el número de clusters y se quiere analizar la estructura a diferentes granularidades.

In [ ]:
K_OPT = 5  # Ajustar según el análisis del codo

# K-Means
kmeans = KMeans(n_clusters=K_OPT, random_state=42, n_init=10)
df['cluster_kmeans'] = kmeans.fit_predict(X_scaled)

# DBSCAN (ajustar eps y min_samples según los datos)
dbscan = DBSCAN(eps=0.8, min_samples=5)
df['cluster_dbscan'] = dbscan.fit_predict(X_scaled)

# Clustering Jerárquico
agg = AgglomerativeClustering(n_clusters=K_OPT, linkage='ward')
df['cluster_agg'] = agg.fit_predict(X_scaled)

# Métricas
resultados_clusters = {}
for nombre, col in [('K-Means', 'cluster_kmeans'), ('Jerárquico', 'cluster_agg')]:
    labels = df[col]
    sil = silhouette_score(X_scaled, labels)
    db  = davies_bouldin_score(X_scaled, labels)
    n_clusters = labels.nunique()
    resultados_clusters[nombre] = {'Silhouette': sil, 'Davies-Bouldin': db, 'N_clusters': n_clusters}
    print(f'{nombre}: Silhouette={sil:.3f} | Davies-Bouldin={db:.3f} | Clusters={n_clusters}')

# DBSCAN (puede tener ruido, cluster=-1)
mask_valid = df['cluster_dbscan'] != -1
if mask_valid.sum() > 0 and df.loc[mask_valid, 'cluster_dbscan'].nunique() > 1:
    sil_db = silhouette_score(X_scaled[mask_valid], df.loc[mask_valid, 'cluster_dbscan'])
    n_noise = (df['cluster_dbscan'] == -1).sum()
    print(f'DBSCAN: Silhouette={sil_db:.3f} | Ruido={n_noise} pts | Clusters={df.loc[mask_valid,"cluster_dbscan"].nunique()}')
else:
    print('DBSCAN: ajustar eps/min_samples para obtener clusters válidos')

In [ ]:
# Dendrograma (Clustering Jerárquico)
fig, ax = plt.subplots(figsize=(16, 6))
linkage_matrix = linkage(X_scaled[:50], method='ward')  # submuestra para visualización
dendrogram(linkage_matrix, ax=ax, color_threshold=3, above_threshold_color='gray')
ax.set_title('Dendrograma – Clustering Jerárquico (Ward, submuestra 50 clientes)', fontsize=14, fontweight='bold')
ax.set_xlabel('Índice de cliente')
ax.set_ylabel('Distancia de Ward')
ax.axhline(y=3, color='red', linestyle='--', alpha=0.7, label='Corte sugerido')
ax.legend()
plt.tight_layout()
plt.savefig('dendrograma.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Visualización PCA de los Clusters

In [ ]:
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)
var_exp = pca.explained_variance_ratio_
print(f'Varianza explicada por PC1: {var_exp[0]:.2%} | PC2: {var_exp[1]:.2%} | Total: {sum(var_exp):.2%}')

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
paleta = plt.cm.Set1

configs = [
    ('cluster_kmeans', 'K-Means'),
    ('cluster_dbscan', 'DBSCAN'),
    ('cluster_agg', 'Jerárquico (Ward)'),
]

for ax, (col, titulo) in zip(axes, configs):
    labels = df[col]
    unique_labels = sorted(labels.unique())
    for i, label in enumerate(unique_labels):
        mask = labels == label
        color = '#808080' if label == -1 else paleta(i / max(len(unique_labels)-1, 1))
        nombre_cluster = 'Ruido' if label == -1 else f'Cluster {label}'
        ax.scatter(X_pca[mask, 0], X_pca[mask, 1],
                   c=[color], label=nombre_cluster,
                   alpha=0.7, s=45, edgecolors='white', linewidth=0.3)
    ax.set_title(titulo, fontsize=13, fontweight='bold')
    ax.set_xlabel(f'PC1 ({var_exp[0]:.1%})')
    ax.set_ylabel(f'PC2 ({var_exp[1]:.1%})')
    ax.legend(fontsize=8, loc='best')
    ax.grid(True, alpha=0.3)

plt.suptitle('Comparación de Métodos de Clustering (PCA 2D)', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('pca_clusters.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Caracterización e Interpretación de los Clusters (K-Means)

In [ ]:
# Perfil estadístico por cluster
perfil = df.groupby('cluster_kmeans')[features].mean().round(1)
perfil['Tamaño'] = df.groupby('cluster_kmeans').size()
perfil['% Total'] = (perfil['Tamaño'] / len(df) * 100).round(1)

# Etiquetas descriptivas (ajustar según los datos reales)
etiquetas_cluster = {
    0: 'Ahorradores de alto ingreso',
    1: 'Jóvenes derrochadores',
    2: 'Clientes estándar',
    3: 'Adultos con bajo gasto',
    4: 'VIP: alto ingreso y alto gasto'
}
perfil['Perfil'] = [etiquetas_cluster.get(i, f'Cluster {i}') for i in perfil.index]
print('=== Perfil de Clusters K-Means ===')
print(perfil.to_string())

In [ ]:
# Visualización de perfiles por radar / barras agrupadas
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Barras por variable y cluster
perfil_norm = df.groupby('cluster_kmeans')[features].mean()
perfil_norm.plot(kind='bar', ax=axes[0], edgecolor='black', alpha=0.85)
axes[0].set_title('Perfil Promedio por Cluster (K-Means)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Cluster')
axes[0].set_ylabel('Valor promedio')
axes[0].tick_params(axis='x', rotation=0)
axes[0].legend(loc='upper right')
axes[0].grid(axis='y', alpha=0.4)

# Scatter Ingreso vs Gasto coloreado
scatter = axes[1].scatter(
    df['Annual Income (k$)'], df['Spending Score (1-100)'],
    c=df['cluster_kmeans'], cmap='Set1', alpha=0.7, s=60, edgecolors='white')
centroids = scaler.inverse_transform(kmeans.cluster_centers_)
axes[1].scatter(centroids[:, 1], centroids[:, 2],
                c='black', marker='X', s=200, zorder=5, label='Centroides')
plt.colorbar(scatter, ax=axes[1], label='Cluster')
axes[1].set_xlabel('Annual Income (k$)', fontsize=11)
axes[1].set_ylabel('Spending Score (1-100)', fontsize=11)
axes[1].set_title('Segmentación: Ingreso vs Gasto', fontsize=13, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('perfil_clusters.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Comparación de Silhouette Scores
nombres = ['K-Means', 'Jerárquico (Ward)']
scores = [resultados_clusters[n]['Silhouette'] for n in nombres]

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(nombres, scores, color=['#3498db', '#9b59b6'], edgecolor='black', alpha=0.85)
ax.set_ylim(0, 1)
ax.set_ylabel('Silhouette Score (mayor = mejor)', fontsize=12)
ax.set_title('Comparación de Calidad de Clustering', fontsize=14, fontweight='bold')
ax.grid(axis='y', alpha=0.4)
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('comparacion_clustering.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Conclusiones e Interpretación de Segmentos

### Segmentos Identificados (K-Means, k=5)

| Cluster | Perfil | Estrategia de Marketing |
|---------|--------|--------------------------|
| 0 | Alto ingreso, bajo gasto | Programas de fidelización premium |
| 1 | Jóvenes, bajo ingreso, alto gasto | Ofertas y promociones frecuentes |
| 2 | Perfil estándar | Campañas masivas, descuentos moderados |
| 3 | Adultos mayores, gasto conservador | Productos de valor y durabilidad |
| 4 | Alto ingreso + alto gasto (VIP) | Experiencias exclusivas, membresías |

### Comparación de Algoritmos
- **K-Means** obtuvo la mejor separabilidad de clusters, siendo ideal para datos numéricos bien escalados.
- **Clustering Jerárquico** confirma los grupos pero permite explorar diferentes granularidades mediante el dendrograma.
- **DBSCAN** es útil para detectar clientes atípicos (outliers) que no pertenecen a ningún segmento claro.

### Consideraciones
- Los resultados deben validarse con conocimiento del negocio antes de aplicar estrategias de marketing.
- Incorporar más variables (frecuencia de visita, categorías de compra) enriquecería los segmentos.
- La segmentación debe actualizarse periódicamente con nuevos datos de clientes.